In [ ]:
%load_ext autoreload
%autoreload 2

# DataLoader

In [ ]:
import os
import sys
import json
from pathlib import Path
import random
import inspect
from pprint import pprint

from dotenv import load_dotenv, find_dotenv
# 1. Locate and load the environment variables from the .env configuration file
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

# 2. Extract configuration variables from the environment
PROJECT_ROOT = os.getenv("PROJECT_ROOT")
REPOS_DIR = os.getenv("REPOS_DIR")
DATA_DIR = os.getenv("DATA_DIR")

project_root = Path(PROJECT_ROOT).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Import your newly structured module
from kg_commit.knowledge.dataloader import CommitDataLoader, JITDatasetAdapter
from kg_commit.knowledge.parsers import FilteredCommitParser, IdentityCommitParser
from kg_commit.knowledge.utils import CommitPayloadPrinter

In [ ]:
# 1. Setup paths
CSV_PATH = f"{DATA_DIR}/apachejit/projects/apache_groovy.csv"

In [ ]:
def generate_repo_map(base_dir: str, prefix: str = "apache/") -> dict[str, str]:
    """
    Scans a base directory for valid git repositories and constructs
    a REPO_MAP dictionary compatible with the CommitDataLoader mapping schema.
    """
    repo_map = {}
    base_path = Path(base_dir)
    
    if not base_path.exists():
        print(f"⚠️ Warning: Base directory '{base_dir}' does not exist.")
        return repo_map

    # Iterate through all direct items in the repos folder
    for item in base_path.iterdir():
        if item.is_dir():
            # Check if it contains a hidden .git directory to verify it's a real repo
            git_dir = item / ".git"
            if git_dir.exists():
                # Reconstruct the project key name (e.g., "apache/groovy")
                project_key = f"{prefix}{item.name.lower()}"
                
                # Assign the absolute string path as the value
                repo_map[project_key] = str(item.resolve())
                
    return repo_map

# Run the auto-generation mapping
REPO_MAP = generate_repo_map(REPOS_DIR, prefix="apache/")

# Print out your freshly discovered mappings
print("📂 Automatically generated REPO_MAP mappings:")
print("-" * 50)
for project, local_path in REPO_MAP.items():
    print(f"  '{project}'")
print("-" * 50)

📂 Automatically generated REPO_MAP mappings:
--------------------------------------------------
  'apache/activemq'
  'apache/camel'
  'apache/cassandra'
  'apache/flink'
  'apache/groovy'
  'apache/hadoop'
  'apache/hadoop-hdfs'
  'apache/hadoop-mapreduce'
  'apache/hbase'
  'apache/hive'
  'apache/ignite'
  'apache/kafka'
  'apache/spark'
  'apache/zeppelin'
  'apache/zookeeper'
--------------------------------------------------


In [ ]:
# 2. Instantiate systems
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)
parser = IdentityCommitParser()

# 3. Pull a random commit record
all_records = list(adapter.stream_records(CSV_PATH))
random_record = random.choice(all_records)

# 4. Extract rich payload parameters from repository metadata
git_payload = loader.fetch_commit_data(
    project=random_record["project"], 
    commit_id=random_record["commit_id"]
)

if git_payload:
    full_payload = {**random_record, **git_payload}
    
    # 5. Print out the raw dictionary dynamically via our class utility
    CommitPayloadPrinter.print_payload(full_payload)
    
    # 6. Parse and check entity configurations
    parsed_results = parser.parse(full_payload)

✅ Success! Raw payload retrieved with customized features.
KEY                       | VALUE
commit_id                 | c8272f2bcba681589b317084b6610384a383ae44
project                   | apache/groovy
buggy                     | False
fix                       | False
year                      | 2019
author_date               | 1552920346
message                   | Support expression::instanceMethod and expression::staticMethod ...
diff                      | [Length: 3758 chars] -> @@ -139,7 +139,7 @@ public class StaticTypesLambdaWriter extends LambdaWriter implements AbstractFun...
parents                   | ['c18fbf174a6a36d514618dcacbc26a1175a7778b']
parents_length            | 1
linked_issues             | []
containing_branches       | ['master']
author_name               | Daniel Sun
author_email              | sunlan@apache.org
authored_timestamp        | 1552920346
authored_datetime         | 2019-03-18T22:45:46+08:00
committer_name            | Paul King
committed_datet

In [ ]:
# 1. Initialize our components
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)

# 2. Test reading records via the Adapter
print("--- Testing CSV Adapter Filtering ---")
records_stream = adapter.stream_records(CSV_PATH)

# Take the first two rows for validation
for i, record in enumerate(records_stream):
    if i >= 2: 
        break
    print(f"\nRecord #{i+1} parsed from CSV:")
    print(record)
    
    # 3. Use the filtered metadata to fetch Git info right away
    print(f"Fetching Git text data for commit: {record['commit_id']}...")
    git_payload = loader.fetch_commit_data(project=record['project'], commit_id=record['commit_id'])
    
    if git_payload:
        print(f"✅ Extracted Message length: {len(git_payload['message'])} chars")
        print(f"✅ Extracted Diff length: {len(git_payload['diff'])} chars")

--- Testing CSV Adapter Filtering ---

Record #1 parsed from CSV:
{'commit_id': '7b8480744ea6e6fb41efd4329bb470c8f3c763db', 'project': 'apache/groovy', 'buggy': 'False', 'fix': 'False', 'year': '2003', 'author_date': '1070355653'}
Fetching Git text data for commit: 7b8480744ea6e6fb41efd4329bb470c8f3c763db...
✅ Extracted Message length: 190 chars
✅ Extracted Diff length: 11929 chars

Record #2 parsed from CSV:
{'commit_id': '192b631e7be302ecde822546ba70a9853ddbda01', 'project': 'apache/groovy', 'buggy': 'False', 'fix': 'False', 'year': '2003', 'author_date': '1063298262'}
Fetching Git text data for commit: 192b631e7be302ecde822546ba70a9853ddbda01...
✅ Extracted Message length: 135 chars
✅ Extracted Diff length: 613 chars


# Gitpython Commit Object

In [ ]:
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)

# Grab a random commit to inspect
record = random.choice(list(adapter.stream_records(CSV_PATH)))
repo = loader._get_repo(record["project"])
commit_obj = repo.commit(record["commit_id"])

print(f"🔬 Dissecting GitPython Commit Object for ID: {commit_obj.hexsha}\n")
print("=" * 60)

properties_list = []
methods_list = []

# Analyze every attribute on the live object
for name, value in inspect.getmembers(commit_obj):
    if name.startswith('_'): 
        continue  # Skip private attributes
        
    try:
        if inspect.ismethod(value) or inspect.isroutine(value):
            methods_list.append(name)
        else:
            properties_list.append((name, type(value).__name__))
    except Exception:
        properties_list.append((name, "Unknown/Unloaded Property"))

print("📋 DATA FIELDS & PROPERTIES AVAILABLE:")
print("-" * 40)
for prop, data_type in sorted(properties_list):
    print(f"  {prop:<25} [Type: {data_type}]")

print("\n⚙️ EXECUTABLE METHODS AVAILABLE:")
print("-" * 40)
for method in sorted(methods_list):
    print(f"  {method}()")

🔬 Dissecting GitPython Commit Object for ID: 7d8857c6aa6b76d1bf3fb9a9a781df94fc3c8f9b

📋 DATA FIELDS & PROPERTIES AVAILABLE:
----------------------------------------
  INDEX                     [Type: DiffConstants]
  Index                     [Type: DiffConstants]
  NULL_BIN_SHA              [Type: bytes]
  NULL_HEX_SHA              [Type: str]
  NULL_TREE                 [Type: DiffConstants]
  TIobj_tuple               [Type: _GenericAlias]
  TYPES                     [Type: tuple]
  author                    [Type: Actor]
  author_tz_offset          [Type: int]
  authored_date             [Type: int]
  authored_datetime         [Type: datetime]
  binsha                    [Type: bytes]
  co_authors                [Type: list]
  committed_date            [Type: int]
  committed_datetime        [Type: datetime]
  committer                 [Type: Actor]
  committer_tz_offset       [Type: int]
  conf_encoding             [Type: str]
  data_stream               [Type: OStream]
  default

# Parsers

In [ ]:
adapter = JITDatasetAdapter()
loader = CommitDataLoader(repo_map=REPO_MAP)
parser = IdentityCommitParser()

print(f"Reading records from {CSV_PATH}...")
all_records = list(adapter.stream_records(CSV_PATH))

if not all_records:
    print("❌ No records found in the metadata file.")
else:
    random_record = random.choice(all_records)
    print(f"🎲 Randomly selected commit ID: {random_record['commit_id']} from {random_record['project']}")
    
    print("Extracting payload from local git repository...")
    git_payload = loader.fetch_commit_data(
        project=random_record["project"], 
        commit_id=random_record["commit_id"]
    )
    
    if git_payload:
        full_payload = {**random_record, **git_payload}
        
        print("\n=================== RAW COMMIT DIFF ===================")
        print(full_payload.get("diff", "No diff available for this commit."))
        print("========================================================\n")
        
        print("Executing simultaneous parse cycle...")
        results = parser.parse(full_payload)
        
        print("\n=== Extracted Knowledge Graph Entities (Random Sample) ===")
        print(json.dumps(results, indent=10))
    else:
        print("❌ Could not extract data from the Git repository. Verify your local paths match.")

Reading records from E:/Projects/kgcommit/data/apachejit/projects/apache_groovy.csv...
🎲 Randomly selected commit ID: d2e88eb480b76c8c897b1aa44b57198bda7c54bb from apache/groovy
Extracting payload from local git repository...

=================== RAW COMMIT DIFF ===================
@@ -183,6 +183,16 @@ public class MarkupBuilder extends BuilderSupport {
         return this;
     }
 
+    /**
+     * Prints data in the body of the current tag, escaping XML entities.
+     * For example: <code>mkp.yield('5 &lt; 7')</code>
+     *
+     * @param value an Object whose toString() representation is to be printed
+     */
+    public void yield(Object value) {
+        yield(value.toString());
+    }
+
     /**
      * Prints data in the body of the current tag, escaping XML entities.
      * For example: <code>mkp.yield('5 &lt; 7')</code>
@@ -193,6 +203,16 @@ public class MarkupBuilder extends BuilderSupport {
         yield(value, true);
     }
 
+    /**
+     * Print data in the body of 

# Gitpython Exploration

In [ ]:
import git

# 1. Access the repository
repo = git.Repo(REPO_MAP['apache/hive']) 
commit = repo.head.commit

# 2. Compare the HEAD commit to its parent
if commit.parents:
    diffs = commit.parents[0].diff(commit, create_patch=True)
    
    if len(diffs) > 0:
        d = diffs[0]
        print(f"--- Dynamically Inspecting All Diff Attributes ---")
        
        # 'dir(d)' returns all attributes/methods
        # We filter out private methods (starting with _) to keep it clean
        for attr in dir(d):
            if not attr.startswith('_'):
                try:
                    value = getattr(d, attr)
                    # Only print if it's not a bound method (keeps the output readable)
                    if not callable(value):
                        print(f"{attr:20}: {value}")
                except Exception:
                    print(f"{attr:20}: <Could not access>")
    else:
        print("No changes in this commit.")
else:
    print("Root commit; no parent to diff against.")

--- Dynamically Inspecting All Diff Attributes ---
NULL_BIN_SHA        : b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00'
NULL_HEX_SHA        : 0000000000000000000000000000000000000000
a_blob              : 59cd3c9a24920503afa714e19541f915226a62c4
a_mode              : 33188
a_path              : ql/src/java/org/apache/hadoop/hive/ql/exec/vector/ptf/VectorPTFOperator.java
a_rawpath           : b'ql/src/java/org/apache/hadoop/hive/ql/exec/vector/ptf/VectorPTFOperator.java'
b_blob              : 6d03bb5ea5d177b8cb4b40987ca82c9a617ce987
b_mode              : 33188
b_path              : ql/src/java/org/apache/hadoop/hive/ql/exec/vector/ptf/VectorPTFOperator.java
b_rawpath           : b'ql/src/java/org/apache/hadoop/hive/ql/exec/vector/ptf/VectorPTFOperator.java'
change_type         : None
copied_file         : False
deleted_file        : False
diff                : b'@@ -578,6 +578,10 @@ private static TypeInfo columnVectorTypeToTypeInfo(Type type) {\n   

C:\Users\Behnam\AppData\Local\Temp\ipykernel_14872\1356399266.py:20: DeprecationWarning: Diff.renamed is deprecated, use Diff.renamed_file instead
  value = getattr(d, attr)


In [ ]:
import git

def find_first_rename(repo_path):
    repo = git.Repo(repo_path)
    
    # Iterate through commits starting from the most recent
    for commit in repo.iter_commits():
        if not commit.parents:
            continue
            
        # Get diffs compared to the first parent
        diffs = commit.parents[0].diff(commit)
        
        for d in diffs:
            # Check for Rename type
            if d.a_path != d.b_path:
                print(f"--- Rename Found in Commit {commit.hexsha[:7]} ---")
                print(f"Old Path : {d.a_path}")
                print(f"New Path : {d.b_path}")
                print(f"Similarity Score: {d.score}/100")
                return d # Return the first one found
                
    print("No renames found in recent history.")
    return None

# Usage
rename_diff = find_first_rename(REPO_MAP['apache/hive'])

--- Rename Found in Commit 13f3208 ---
Old Path : .github/workflows/docker-GA-images.yml
New Path : .github/workflows/docker-images.yml
Similarity Score: 92/100


# Fast commit fetch

In [ ]:
# %% [markdown]
# # Portfolio-Wide Performance Test: `fetch_all_commits_fast`
# This notebook scans, profiles, and aggregates Git action metrics across all repositories in your local environment.

# %%
import os
import sys
import time
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# 1. Load system environment paths
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)

PROJECT_ROOT = os.getenv("PROJECT_ROOT")
REPOS_DIR = os.getenv("REPOS_DIR")

# 2. Append project root so Python can find your module
project_root = Path(PROJECT_ROOT).resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# 3. Direct target import
from kg_commit.knowledge.dataloader import CommitDataLoader

# %% [markdown]
# ## Step 1: Map All Local Repositories Dynamically

# %%
def generate_repo_map(base_dir: str, prefix: str = "apache/") -> dict[str, str]:
    """
    Scans a base directory for valid git repositories and constructs
    a REPO_MAP dictionary compatible with the CommitDataLoader mapping schema.
    """
    repo_map = {}
    base_path = Path(base_dir)
    
    if not base_path.exists():
        print(f"⚠️ Warning: Base directory '{base_dir}' does not exist.")
        return repo_map

    for item in base_path.iterdir():
        if item.is_dir():
            git_dir = item / ".git"
            if git_dir.exists():
                project_key = f"{prefix}{item.name.lower()}"
                repo_map[project_key] = str(item.resolve())
                
    return repo_map

# Run the auto-generation mapping using your environment variables
REPO_MAP = generate_repo_map(REPOS_DIR, prefix="apache/")

print("📂 AUTOMATICALLY GENERATED DISCOVERY SCHEMA MAPPINGS:")
print("-" * 60)
for project, local_path in REPO_MAP.items():
    print(f"  • Found -> Key: '{project}' at {local_path}")
print("-" * 60)

# %% [markdown]
# ## Step 2: Sequential Engine Execution & Statistical Aggregation

# %%
# Initialize your data loader with all discovered maps
data_loader = CommitDataLoader(repo_map=REPO_MAP)

# Portfolio metrics trackers
portfolio_results = {}
global_start_time = time.time()

print("🚀 COMMENCING PORTFOLIO-WIDE PARSING STREAMS...")
print("=" * 80)
# %% [markdown]
# ## Step 2: Sequential Engine Execution & Statistical Aggregation

for project_key in REPO_MAP.keys():
    print(f"\n⏳ Processing stream for target project: '{project_key}'...")
    project_start = time.time()
    
    try:
        # Force generator execution to fully unpack commits into memory
        commits_pool = list(data_loader.fetch_all_commits_fast(project=project_key, limit=-1))
        project_duration = time.time() - project_start
        
        # Core metric counters for this individual repository
        stats = {
            "total_commits": len(commits_pool),
            "duration": project_duration,
            "commits_A": 0, "commits_M": 0, "commits_R": 0, "commits_C": 0, "commits_D": 0,
            "files_A": 0, "files_M": 0, "files_R": 0, "files_C": 0, "files_D": 0
        }
        
        # Helper to ensure we pull exactly ONE rename sample string per project
        project_rename_sample = None
        
        for c in commits_pool:
            num_A = len(c.get("files_added_list", []))
            num_M = len(c.get("files_modified_list", []))
            num_R = len(c.get("files_renamed_list", []))
            num_C = len(c.get("files_copied_list", []))
            num_D = len(c.get("files_deleted_list", []))
            
            # Capture the very first valid file rename example we encounter
            if num_R > 0 and project_rename_sample is None:
                # Store a tuple format: (old_path, new_path)
                project_rename_sample = c.get("files_renamed_list", [])[0]
            
            # Aggregate total files touched
            stats["files_A"] += num_A
            stats["files_M"] += num_M
            stats["files_R"] += num_R
            stats["files_C"] += num_C
            stats["files_D"] += num_D
            
            # Count commits containing specific change actions
            if num_A > 0: stats["commits_A"] += 1
            if num_M > 0: stats["commits_M"] += 1
            if num_R > 0: stats["commits_R"] += 1
            if num_C > 0: stats["commits_C"] += 1
            if num_D > 0: stats["commits_D"] += 1
            
        portfolio_results[project_key] = stats
        
        # Display isolated telemetry statistics for this specific repo block
        throughput = stats["total_commits"] / project_duration if project_duration > 0 else 0
        print(f"📊 INTERMEDIATE RESULTS FOR '{project_key}':")
        print(f"  ├── Elapsed Time : {project_duration:.3f}s | Speed: {throughput:.2f} commits/sec")
        print(f"  ├── Commits with -> A: {stats['commits_A']:<5} M: {stats['commits_M']:<5} R: {stats['commits_R']:<5} C: {stats['commits_C']:<5} D: {stats['commits_D']}")
        print(f"  └── Total Files  -> A: {stats['files_A']:<5} M: {stats['files_M']:<5} R: {stats['files_R']:<5} C: {stats['files_C']:<5} D: {stats['files_D']}")
        
        # Inject the live sample print statement directly below the file counters
        if project_rename_sample:
            print(f"  └── 🟨 RENAME SAMPLE: {project_rename_sample[0]} ➔ {project_rename_sample[1]}")
        else:
            print(f"  └── 🟨 RENAME SAMPLE: None found matching structural .java extension rules.")
            
    except Exception as e:
        print(f"❌ Critical Failure streaming project '{project_key}': {str(e)}")
        continue

global_duration = time.time() - global_start_time
print("\n" + "=" * 80)
print(f"🏁 Portfolio stream evaluation completed in {global_duration:.3f} seconds.")
print("=" * 80)

global_duration = time.time() - global_start_time

# %% [markdown]
# ## Step 3: Global Portfolio Summary Report

# %%
print("\n" + "=" * 90)
print("                       GLOBAL PORTFOLIO ANALYSIS SUMMARY                        ")
print("=" * 90)
print(f"{'PROJECT KEY':<25} | {'COMMITS':<7} | {'TIME':<7} | {'SPEED (C/s)':<11} | {'FILES MODIFIED (A/M/R/C/D)':<25}")
print("-" * 90)

grand_total_commits = 0
grand_total_files = 0

for proj, data in portfolio_results.items():
    thru = data["total_commits"] / data["duration"] if data["duration"] > 0 else 0
    file_breakdown = f"{data['files_A']}/{data['files_M']}/{data['files_R']}/{data['files_C']}/{data['files_D']}"
    
    grand_total_commits += data["total_commits"]
    grand_total_files += (data['files_A'] + data['files_M'] + data['files_R'] + data['files_C'] + data['files_D'])
    
    print(f"{proj:<25} | {data['total_commits']:<7} | {data['duration']:5.2f}s | {thru:<11.2f} | {file_breakdown:<25}")

print("-" * 90)
print(f"Portfolio Totals: {len(portfolio_results)} Projects checked.")
print(f"Total Combined Java Commits Extracted : {grand_total_commits}")
print(f"Total Combined Java Files Processed   : {grand_total_files}")
print(f"Total Pipeline Execution Wall Time   : {global_duration:.3f} seconds")
print("=" * 90)

📂 AUTOMATICALLY GENERATED DISCOVERY SCHEMA MAPPINGS:
------------------------------------------------------------
  • Found -> Key: 'apache/activemq' at E:\repos\activemq
  • Found -> Key: 'apache/camel' at E:\repos\camel
  • Found -> Key: 'apache/cassandra' at E:\repos\cassandra
  • Found -> Key: 'apache/flink' at E:\repos\flink
  • Found -> Key: 'apache/groovy' at E:\repos\groovy
  • Found -> Key: 'apache/hadoop' at E:\repos\hadoop
  • Found -> Key: 'apache/hadoop-hdfs' at E:\repos\hadoop-hdfs
  • Found -> Key: 'apache/hadoop-mapreduce' at E:\repos\hadoop-mapreduce
  • Found -> Key: 'apache/hbase' at E:\repos\hbase
  • Found -> Key: 'apache/hive' at E:\repos\hive
  • Found -> Key: 'apache/ignite' at E:\repos\ignite
  • Found -> Key: 'apache/kafka' at E:\repos\kafka
  • Found -> Key: 'apache/spark' at E:\repos\spark
  • Found -> Key: 'apache/zeppelin' at E:\repos\zeppelin
  • Found -> Key: 'apache/zookeeper' at E:\repos\zookeeper
-------------------------------------------------------

In [ ]:
# Select a repository that is guaranteed to have all action types (e.g., apache/groovy)
raw_log_stream = repo.git.log("--reverse", f"--format={log_format}", "--numstat", "--summary", "-n 1500")
raw_blocks = [b for b in raw_log_stream.split(delimiter) if b.strip()]

sample_add_block = None
sample_modify_block = None
sample_delete_block = None
sample_rename_block = None

# Scan the raw blocks for literal git string signatures
for block in raw_blocks:
    # Look for a pure modification (has text but no structural summary lines)
    if "\t" in block and not any(x in block for x in ["create mode", "delete mode", "rename "]):
        if not sample_modify_block and ".java" in block:
            sample_modify_block = block
            
    # Look for an explicit file creation
    if "create mode" in block and not sample_add_block and ".java" in block:
        sample_add_block = block
        
    # Look for an explicit file deletion
    if "delete mode" in block and not sample_delete_block and ".java" in block:
        sample_delete_block = block
        
    # Look for an explicit file rename/move
    if "rename " in block and not sample_rename_block and ".java" in block:
        sample_rename_block = block

# --- PRINTING THE RAW UNTOUCHED GIT LOG BLOCKS ---

if sample_add_block:
    print("================================================================================")
    print(" 🟥 RAW GIT STREAM OUTPUT FOR AN ADDITION ('A')                                 ")
    print("================================================================================")
    print(sample_add_block.strip())
    print("================================================================================\n")

if sample_modify_block:
    print("================================================================================")
    print(" 🟪 RAW GIT STREAM OUTPUT FOR A MODIFICATION ('M')                             ")
    print("================================================================================")
    print(sample_modify_block.strip())
    print("================================================================================\n")

if sample_delete_block:
    print("================================================================================")
    print(" 🟥 RAW GIT STREAM OUTPUT FOR A DELETION ('D')                                 ")
    print("================================================================================")
    print(sample_delete_block.strip())
    print("================================================================================\n")

if sample_rename_block:
    print("================================================================================")
    print(" 🟨 RAW GIT STREAM OUTPUT FOR A RENAME ('R')                                   ")
    print("================================================================================")
    print(sample_rename_block.strip())
    print("================================================================================")

 🟥 RAW GIT STREAM OUTPUT FOR AN ADDITION ('A')                                 
8873fa5c3e268f4e164695c463fd86fc15e50830|Eric Milles|eric.milles@thomsonreuters.com|Eric Milles|1728134919|1728134919|GROOVY-11451: add test case

7	0	src/test/groovy/transform/stc/ConstructorsSTCTest.groovy
8	0	src/test/groovy/transform/stc/Pojo11451.java
 create mode 100644 src/test/groovy/transform/stc/Pojo11451.java

 🟪 RAW GIT STREAM OUTPUT FOR A MODIFICATION ('M')                             
35ee6762510b11eb527d1df4d857af68d8c0876a|Eric Milles|eric.milles@thomsonreuters.com|Eric Milles|1725981263|1725981449|GROOVY-11366: STC: implicit-`this` closure field call

10	5	src/main/java/org/codehaus/groovy/transform/stc/StaticTypeCheckingVisitor.java
41	14	src/test/groovy/transform/stc/ClosuresSTCTest.groovy

 🟥 RAW GIT STREAM OUTPUT FOR A DELETION ('D')                                 
cef611c2d22e76be5a8b3e1cc73d3774a3a997e9|Eric Milles|eric.milles@thomsonreuters.com|Eric Milles|1729019844|1729019844|GROO

# Fast Commit Work

In [29]:
import os
import sys
import re
import datetime
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from git import Repo

# 1. Load variables from your local environment configuration file
load_dotenv(find_dotenv())
PROJECT_ROOT = os.getenv("PROJECT_ROOT")
REPOS_DIR = os.getenv("REPOS_DIR")

# 2. Target the specific local repository directory path manually
# Swap 'apache/groovy' or the path if testing a different repository folder
TARGET_PROJECT = "apache/groovy"
LOCAL_REPO_PATH = Path(REPOS_DIR) / "groovy"

print(f"Opening Git repository at: {LOCAL_REPO_PATH.resolve()}")
repo = Repo(str(LOCAL_REPO_PATH))

# 3. Spool the low-level git log output directly into a raw text block
delimiter = "||--NEXT_COMMIT--||"
log_format = f"{delimiter}%H|%aN|%aE|%cN|%at|%ct|%B"

print("⏳ Spooling raw logs with file numstats...")
raw_log_stream = repo.git.log("--reverse", f"--format={log_format}", "--numstat", "-n 5")
raw_commits = [c for c in raw_log_stream.split(delimiter) if c.strip()]

print(f"✅ Captured {len(raw_commits)} raw commit blocks to inspect manually!")

Opening Git repository at: E:\repos\groovy
⏳ Spooling raw logs with file numstats...
✅ Captured 5 raw commit blocks to inspect manually!


In [30]:
# Let's peek at the very first raw block inside your spool array
sample_block = raw_commits[0]

print("🔍 RAW GIT LOG FORMAT INSPECTION:")
print("=" * 80)
print(sample_block)
print("=" * 80)

🔍 RAW GIT LOG FORMAT INSPECTION:
67785389d22431d721cd843ebcf744d1dced505a|Daniel Sun|sunlan@apache.org|Daniel Sun|1777218638|1777218638|Add missing Javadoc for groovy-csv and groovy-markdown


16	0	subprojects/groovy-csv/src/main/java/groovy/csv/CsvBuilder.java
16	0	subprojects/groovy-csv/src/main/java/groovy/csv/CsvRuntimeException.java
4	0	subprojects/groovy-csv/src/main/java/groovy/csv/CsvSlurper.java
10	0	subprojects/groovy-markdown/src/main/java/groovy/markdown/MarkdownDocument.java
16	0	subprojects/groovy-markdown/src/main/java/groovy/markdown/MarkdownRuntimeException.java
40	0	subprojects/groovy-markdown/src/main/java/groovy/markdown/MarkdownSlurper.java
5	0	subprojects/groovy-markdown/src/main/java/groovy/markdown/TableSupport.java



In [23]:
# Setup your regular expression engine patterns
issue_pattern = re.compile(r'\b([A-Z]+-\d+|#\d+|GH-\d+)\b')

print("🛠️ BREAKING DOWN THE STRIPPED LINES:")
print("-" * 50)

# 1. Isolate lines and inspect the metadata header row
lines = sample_block.strip().split("\n")
header = lines[0].split("|")

commit_id = header[0]
author_name = header[1]
author_email = header[2]
committer_name = header[3]
authored_ts = int(header[4])
committed_ts = int(header[5])

print(f"[HEADER PARSED]")
print(f" -> SHA: {commit_id}")
print(f" -> Author: {author_name} ({author_email})")
print(f" -> Committer: {committer_name}")

# 2. Convert Unix timestamps to readable ISO formatted values
authored_dt = datetime.datetime.fromtimestamp(authored_ts, datetime.timezone.utc).isoformat()
committed_dt = datetime.datetime.fromtimestamp(committed_ts, datetime.timezone.utc).isoformat()
print(f" -> Authored Datetime: {authored_dt}")

# 3. Cleanly separate message lines from the tab-delimited file changes
message_lines = [header[6]]
numstat_lines = []

for line in lines[1:]:
    if "\t" in line:
        numstat_lines.append(line)
    else:
        message_lines.append(line)

message_body = "\n".join(message_lines).strip()
print(f" -> Clean Message Extract: \"{message_body.splitlines()[0][:60]}...\"")

# 4. Parse file insertions, deletions, and paths manually
files_added = []
files_modified = []
files_deleted = []
files_renamed_details = []

java_insertions = 0
java_deletions = 0
max_directory_depth = 0

print(f"\n[NUMSTAT FILE BLOCK PARSING] - Total numstat lines: {len(numstat_lines)}")

for line in numstat_lines:
    parts = line.split("\t")
    added_str, deleted_str, filepath = parts[0], parts[1], parts[2]
    
    # Filter scope out immediately if it's not a source file
    if not filepath.lower().endswith('.java'):
        continue
        
    add_val = int(added_str) if added_str.isdigit() else 0
    del_val = int(deleted_str) if deleted_str.isdigit() else 0
    java_insertions += add_val
    java_deletions += del_val
    
    # Calculate file system structural depths
    depth = len(Path(filepath).parts) - 1
    if depth > max_directory_depth:
        max_directory_depth = depth
        
    # Isolate paths or catch renames
    if " => " in filepath:
        match = re.search(r'\{(.*?) => (.*?)\}', filepath)
        if match:
            old_part, new_part = match.group(1), match.group(2)
            old_path = filepath.replace(match.group(0), old_part).replace("//", "/")
            new_path = filepath.replace(match.group(0), new_part).replace("//", "/")
            files_renamed_details.append((old_path, new_path))
        else:
            r_parts = filepath.split(" => ")
            files_renamed_details.append((r_parts[0], r_parts[1]))
    elif add_val > 0 and del_val == 0:
        files_added.append(filepath)
    elif del_val > 0 and add_val == 0:
        files_deleted.append(filepath)
    else:
        files_modified.append(filepath)

# 5. Review calculated metrics aggregation
print(f" -> Accumulated Java Additions: {java_insertions} lines")
print(f" -> Accumulated Java Deletions: {java_deletions} lines")
print(f" -> Verified Maximum Depth    : {max_directory_depth}")
print(f" -> Java Files Added List     : {len(files_added)} files")
print(f" -> Java Files Modified List  : {len(files_modified)} files")
print(f" -> Java Files Deleted List   : {len(files_deleted)} files")
print(f" -> Java Renames Tracked      : {len(files_renamed_details)} paths")

🛠️ BREAKING DOWN THE STRIPPED LINES:
--------------------------------------------------
[HEADER PARSED]
 -> SHA: c1e892cd7d924eb4a4b3326fa547adf10f7873e4
 -> Author: Paul King (paulk@asert.com.au)
 -> Committer: Paul King
 -> Authored Datetime: 2026-04-21T03:25:55+00:00
 -> Clean Message Extract: "update doco..."

[NUMSTAT FILE BLOCK PARSING] - Total numstat lines: 1
 -> Accumulated Java Additions: 0 lines
 -> Accumulated Java Deletions: 0 lines
 -> Verified Maximum Depth    : 0
 -> Java Files Added List     : 0 files
 -> Java Files Modified List  : 0 files
 -> Java Files Deleted List   : 0 files
 -> Java Renames Tracked      : 0 paths


In [26]:
# Select the very first commit block from the spooled stream
sample_block = raw_commits[0]

print("================================================================================")
print("                       RAW COMMIT BLOCK STREAM ENTRY                            ")
print("================================================================================")
print(sample_block)
print("================================================================================\n")

# Process lines exactly like the inner loop logic
lines = sample_block.strip().split("\n")
header_line = lines[0]
header_parts = header_line.split("|")

print("================================================================================")
print(" 1. METADATA HEADER LINE SPLIT                                                  ")
print("================================================================================")
print(f"Raw Header String : {header_line}")
print("-" * 80)
for idx, part in enumerate(header_parts):
    print(f"  [{idx}] Field Token -> {repr(part)}")
print("================================================================================\n")

# Reconstruct message strings vs numstat strings sequentially
message_lines = [header_parts[5]] if len(header_parts) > 5 else []
numstat_lines = []

for line in lines[1:]:
    if "\t" in line:
        numstat_lines.append(line)
    else:
        message_lines.append(line)

print("================================================================================")
print(" 2. SEPARATED MESSAGE LINES                                                     ")
print("================================================================================")
print(f"Total lines grouped as message text: {len(message_lines)}")
print("-" * 80)
for idx, m_line in enumerate(message_lines):
    print(f"  Line {idx}: {repr(m_line)}")
print("================================================================================\n")

print("================================================================================")
print(" 3. SEPARATED RAW NUMSTAT FILE ENTRIES                                          ")
print("================================================================================")
print(f"Total lines grouped as tab-separated file records: {len(numstat_lines)}")
print("-" * 80)
for idx, n_line in enumerate(numstat_lines):
    # Use repr() explicitly to show literal '\t' characters visually
    parts = n_line.split("\t")
    print(f"  Line {idx}: {repr(n_line)}")
    print(f"    └─ Parsed -> Insertions: {parts[0]} | Deletions: {parts[1]} | Filepath: {parts[2]}")
print("================================================================================")

                       RAW COMMIT BLOCK STREAM ENTRY                            
67785389d22431d721cd843ebcf744d1dced505a|Daniel Sun|sunlan@apache.org|Daniel Sun|1777218638|1777218638|Add missing Javadoc for groovy-csv and groovy-markdown


16	0	subprojects/groovy-csv/src/main/java/groovy/csv/CsvBuilder.java
16	0	subprojects/groovy-csv/src/main/java/groovy/csv/CsvRuntimeException.java
4	0	subprojects/groovy-csv/src/main/java/groovy/csv/CsvSlurper.java
10	0	subprojects/groovy-markdown/src/main/java/groovy/markdown/MarkdownDocument.java
16	0	subprojects/groovy-markdown/src/main/java/groovy/markdown/MarkdownRuntimeException.java
40	0	subprojects/groovy-markdown/src/main/java/groovy/markdown/MarkdownSlurper.java
5	0	subprojects/groovy-markdown/src/main/java/groovy/markdown/TableSupport.java


 1. METADATA HEADER LINE SPLIT                                                  
Raw Header String : 67785389d22431d721cd843ebcf744d1dced505a|Daniel Sun|sunlan@apache.org|Daniel Sun|1777218638|1777218

In [33]:
### Notebook Verification Cell

import re
import datetime
from pathlib import Path

# 1. Update the git log spooling command to include the '--summary' instruction
delimiter = "||--NEXT_COMMIT--||"
log_format = f"{delimiter}%H|%aN|%aE|%cN|%at|%ct|%B"

print("⏳ Spooling raw logs with combined numstats and summary data...")
# We grab a slightly larger pool (-n 20) to ensure we intercept diverse actions like creations or renames
raw_log_stream = repo.git.log("--reverse", f"--format={log_format}", "--numstat", "--summary", "-n 20")
raw_commits = [c for c in raw_log_stream.split(delimiter) if c.strip()]

# 2. Pick a rich commit block to inspect manually
# (You can shift the index if you want to inspect a different commit block in the batch)
sample_block = raw_commits[0]

print("================================================================================")
print("                       RAW OBJECT STREAM WITH SUMMARY                           ")
print("================================================================================")
print(sample_block)
print("================================================================================\n")

# 3. Simulate parsing execution exactly like the class method
lines = sample_block.strip().split("\n")
header_parts = lines[0].split("|")

commit_id = header_parts[0]
author_name = header_parts[1]
author_email = header_parts[2]
committer_name = header_parts[3]
authored_ts = int(header_parts[4])
committed_ts = int(header_parts[5])

# Segregate code paragraph documentation lines from raw data tokens
message_lines = [header_parts[6]] if len(header_parts) > 6 else []
data_lines = []
for line in lines[1:]:
    if "\t" in line or line.strip().startswith(('create mode', 'delete mode', 'rename', 'copy')):
        data_lines.append(line.strip())
    else:
        message_lines.append(line)

print("================================================================================")
print(" 1. DATA LINE CLASSIFICATION BREAKDOWN                                          ")
print("================================================================================")
print(f"Total structured lines pulled from commit body: {len(data_lines)}")
print("-" * 80)

# SUB-PASS 1: Build the Line Tally Map
numstat_map = {}
for line in data_lines:
    if "\t" in line:
        parts = line.split("\t")
        if len(parts) >= 3:
            numstat_map[parts[2]] = (parts[0], parts[1])
            print(f" [NUMSTAT FOUND]  -> File: {parts[2]} (+{parts[0]} / -{parts[1]})")

print("-" * 80)

# SUB-PASS 2 & 3: Run the Multi-Stage Categorizer
files_added = []
files_deleted = []
files_modified = []
files_renamed_details = []
files_copied_list = []
processed_raw_paths = set()

for line in data_lines:
    if line.startswith('create mode'):
        filepath = line.split(' ', 3)[-1]
        print(f" 🟩 [SUMMARY MATCH] -> DETECTED CHANGE_TYPE 'A' (Create) via: '{line}'")
        if filepath.lower().endswith('.java') and filepath in numstat_map:
            files_added.append(filepath)
            processed_raw_paths.add(filepath)

    elif line.startswith('delete mode'):
        filepath = line.split(' ', 3)[-1]
        print(f" 🟥 [SUMMARY MATCH] -> DETECTED CHANGE_TYPE 'D' (Delete) via: '{line}'")
        if filepath.lower().endswith('.java') and filepath in numstat_map:
            files_deleted.append(filepath)
            processed_raw_paths.add(filepath)

    elif line.startswith('rename '):
        raw_path_block = line.split(' ', 1)[1].rsplit(' (', 1)[0]
        print(f" 🟨 [SUMMARY MATCH] -> DETECTED CHANGE_TYPE 'R' (Rename) via: '{line}'")
        if raw_path_block in numstat_map:
            processed_raw_paths.add(raw_path_block)
            if " => " in raw_path_block:
                match = re.search(r'\{(.*?) => (.*?)\}', raw_path_block)
                if match:
                    old_part, new_part = match.group(1), match.group(2)
                    old_path = raw_path_block.replace(match.group(0), old_part).replace("//", "/")
                    new_path = raw_path_block.replace(match.group(0), new_part).replace("//", "/")
                else:
                    r_parts = raw_path_block.split(" => ")
                    old_path, new_path = r_parts[0], r_parts[1]
                if new_path.lower().endswith('.java'):
                    files_renamed_details.append((old_path, new_path))

    elif line.startswith('copy '):
        raw_path_block = line.split(' ', 1)[1].rsplit(' (', 1)[0]
        print(f" 🟦 [SUMMARY MATCH] -> DETECTED CHANGE_TYPE 'C' (Copy) via: '{line}'")
        if raw_path_block in numstat_map:
            processed_raw_paths.add(raw_path_block)
            if " => " in raw_path_block:
                match = re.search(r'\{(.*?) => (.*?)\}', raw_path_block)
                new_path = raw_path_block.replace(match.group(0), match.group(2)).replace("//", "/") if match else raw_path_block.split(" => ")[1]
                if new_path.lower().endswith('.java'):
                    files_copied_list.append(new_path)

# Fallbacks left inside the pool default down to Modifications ('M')
for raw_filepath, (added_str, deleted_str) in numstat_map.items():
    if raw_filepath in processed_raw_paths:
        continue
    
    clean_path = raw_filepath
    if " => " in raw_filepath:
        match = re.search(r'\{(.*?) => (.*?)\}', raw_filepath)
        clean_path = raw_filepath.replace(match.group(0), match.group(2)).replace("//", "/") if match else raw_filepath.split(" => ")[1]
        
    if clean_path.lower().endswith('.java'):
        print(f" 🟪 [FALLBACK MATCH] -> DETECTED CHANGE_TYPE 'M' (Modify) via leftover numstat: '{raw_filepath}'")
        files_modified.append(clean_path)

print("====================================================================")
print(" 2. PARSED PAYLOAD DATA EVALUATION                                  ")
print("====================================================================")
print(f"Commit ID Hash       : {commit_id}")
print(f"Author Name          : {author_name}")
print(f"Committer Name       : {committer_name}")
print(f"Files Added ('A')    : {files_added}")
print(f"Files Deleted ('D')  : {files_deleted}")
print(f"Files Modified ('M') : {files_modified}")
print(f"Files Renamed ('R')  : {files_renamed_details}")
print(f"Files Copied ('C')   : {files_copied_list}")
print("====================================================================")

⏳ Spooling raw logs with combined numstats and summary data...
                       RAW OBJECT STREAM WITH SUMMARY                           
fb7e1ac0905014c9162a1bc1a700e0296337ca15|Daniel Sun|sunlan@apache.org|Daniel Sun|1777177601|1777177601|Add missing Javadoc for groovy-xml


149	0	subprojects/groovy-xml/src/main/groovy/groovy/xml/Entity.groovy
67	38	subprojects/groovy-xml/src/main/groovy/groovy/xml/StaxBuilder.groovy
17	0	subprojects/groovy-xml/src/main/groovy/groovy/xml/StreamingDOMBuilder.groovy
14	0	subprojects/groovy-xml/src/main/groovy/groovy/xml/StreamingMarkupBuilder.groovy
16	0	subprojects/groovy-xml/src/main/groovy/groovy/xml/StreamingSAXBuilder.groovy
6	0	subprojects/groovy-xml/src/main/groovy/groovy/xml/XmlParserFactory.groovy
6	0	subprojects/groovy-xml/src/main/groovy/groovy/xml/XmlSlurperFactory.groovy
11	1	subprojects/groovy-xml/src/main/groovy/groovy/xml/streamingmarkupsupport/AbstractStreamingBuilder.groovy
78	0	subprojects/groovy-xml/src/main/java/groovy/xml/DO